In [ ]:
model_sales, scaler_sales = run_regression(df_model, 'SalesAmount')
model_qty, scaler_qty = run_regression(df_model, 'OrderQuantity')

In [ ]:
def run_regression(df, target):
    X = df.drop(columns=['SalesAmount', 'OrderQuantity'])
    y = df[target]
    
    # Scale features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, random_state=42
    )
    
    # Sklearn model
    model = LinearRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    print(f"\n{'='*40}")
    print(f"Target: {target}")
    print(f"R² Score:  {r2_score(y_test, y_pred):.4f}")
    print(f"RMSE:      {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
    
    # Statsmodels for p-values
    X_sm = sm.add_constant(X_scaled)
    sm_model = sm.OLS(y, X_sm).fit()
    print(sm_model.summary())
    
    # Feature importance plot
    coef_df = pd.DataFrame({
        'Feature': X.columns,
        'Coefficient': model.coef_
    }).sort_values('Coefficient', ascending=False)
    
    plt.figure(figsize=(10, 6))
    sns.barplot(data=coef_df, x='Coefficient', y='Feature', palette='coolwarm')
    plt.title(f'Feature Coefficients — {target}')
    plt.tight_layout()
    plt.savefig(f'regression_{target}.png')
    plt.show()
    
    return model, scaler

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(df_model.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.savefig('correlation_heatmap.png')
plt.show()

In [ ]:
# Extract month and year from OrderDateKey (format: YYYYMMDD)
df['Year'] = df['OrderDateKey'].astype(str).str[:4].astype(int)
df['Month'] = df['OrderDateKey'].astype(str).str[4:6].astype(int)

# Encode Source column
df['IsInternet'] = (df['Source'] == 'Internet').astype(int)

# Drop original columns not needed for regression
df_model = df.drop(columns=['OrderDateKey', 'Source'])

print(df_model.dtypes)
df_model.describe()

In [ ]:
conn = sqlite3.connect('AdventureWorks.db')

internet = pd.read_sql("""
    SELECT 
        s.SalesAmount,
        s.OrderQuantity,
        s.UnitPrice,
        s.DiscountAmount,
        s.TotalProductCost,
        s.TaxAmt,
        s.Freight,
        s.UnitPriceDiscountPct,
        s.OrderDateKey,
        p.PromotionKey,
        t.SalesTerritoryKey,
        'Internet' as Source
    FROM FactInternetSales s
    LEFT JOIN DimPromotion p ON s.PromotionKey = p.PromotionKey
    LEFT JOIN DimSalesTerritory t ON s.SalesTerritoryKey = t.SalesTerritoryKey
""", conn)

reseller = pd.read_sql("""
    SELECT 
        s.SalesAmount,
        s.OrderQuantity,
        s.UnitPrice,
        s.DiscountAmount,
        s.TotalProductCost,
        s.TaxAmt,
        s.Freight,
        s.UnitPriceDiscountPct,
        s.OrderDateKey,
        p.PromotionKey,
        t.SalesTerritoryKey,
        'Reseller' as Source
    FROM FactResellerSales s
    LEFT JOIN DimPromotion p ON s.PromotionKey = p.PromotionKey
    LEFT JOIN DimSalesTerritory t ON s.SalesTerritoryKey = t.SalesTerritoryKey
""", conn)

df = pd.concat([internet, reseller], ignore_index=True)
conn.close()
print(df.shape)
df.head()

In [ ]:
# Extract month and year from OrderDateKey (format: YYYYMMDD)
df['Year'] = df['OrderDateKey'].astype(str).str[:4].astype(int)
df['Month'] = df['OrderDateKey'].astype(str).str[4:6].astype(int)

# Encode Source column
df['IsInternet'] = (df['Source'] == 'Internet').astype(int)

# Drop original columns not needed for regression
df_model = df.drop(columns=['OrderDateKey', 'Source'])

print(df_model.dtypes)
df_model.describe()

In [1]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')